In [0]:
%python
from pyspark.sql.functions import col, count, sum as _sum, round as _round, coalesce, lit

enc = spark.read.table("meridian_dev.silver.encounters")
payers = (spark.read.table("meridian_dev.bronze.payers")
    .select(col("Id").alias("payer_id"), col("NAME").alias("payer_name")))

encounters_by_payer = (enc
    .join(payers, on="payer_id", how="left")
    .withColumn("payer_name", coalesce(col("payer_name"), lit("Self-Pay/Unknown")))
    .groupBy("payer_name")
    .agg(
        count("*").alias("num_encounters"),
        _round(_sum("total_cost"), 2).alias("total_cost"),
        _round(_sum("payer_coverage"), 2).alias("total_covered"),
        _round(_sum("payer_coverage") / _sum("total_cost") * 100, 1).alias("coverage_pct"),
    )
    .orderBy(col("num_encounters").desc()))

encounters_by_payer.write.format("delta").mode("overwrite") \
    .saveAsTable("meridian_dev.gold.encounters_by_payer")

print("Gold encounters_by_payer built.")

In [0]:
SELECT payer_name, num_encounters, total_cost, total_covered, coverage_pct
FROM meridian_dev.gold.encounters_by_payer
ORDER BY num_encounters DESC;